<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Waterloo-Park-LiDAR-Tree-Detection-Pipeline
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A Machine Learning Exploration of LiDAR Classification
</h2>
</div>

# Module 1: *Data Exploration*
##### Version Number: 1.0
---
### Contents  
---
### Notes
---
### Inputs
---
### Outputs  
---
### User Created Dependencies  
---
### Third Party Dependencies

In [142]:
import pandas as pd
import numpy as np
import laspy
import rasterio
from rasterio.transform import rowcol

### Load LAS file

In [171]:
# Load the LAS/LAZ file
las = laspy.read("data/clipped_point_cloud.las")

cloud = pd.DataFrame({
    "X": np.array(las.x),
    "Y": np.array(las.y),
    "Z": np.array(las.z),
    "intensity": np.array(las.intensity),
    "classification": np.array(las.classification)
})

cloud.head()

point_ids = np.arange(len(cloud))

### Load Orthophoto Raster

In [172]:
with rasterio.open("data/clipped_ortho.tif") as src:
    transform = src.transform
    bounds = src.bounds
    crs = src.crs
    
    x_min = bounds.left
    y_min = bounds.bottom
    x_max = bounds.right
    y_max = bounds.top

    width = src.width
    height = src.height

    red_band = src.read(1)  # Red
    green_band = src.read(2)  # Green
    blue_band = src.read(3)  # Blue

In [192]:
ground = cloud[cloud.classification == 2]

In [193]:
gx = ground.X
gy = ground.Y
gz = ground.Z

In [194]:
with rasterio.open("data/clipped_ortho.tif") as src:
    transform = src.transform
    width = src.width
    height = src.height

In [195]:
from rasterio.transform import xy

cols, rows = np.meshgrid(
    np.arange(width),
    np.arange(height)
)

xs, ys = xy(transform, rows, cols)
xs = np.array(xs)
ys = np.array(ys)

In [196]:
from scipy.interpolate import griddata

dtm = griddata(
    points=(gx, gy),
    values=gz,
    xi=(xs, ys),
    method="nearest"  # or "linear"
)

In [199]:
dtm.shape

(74360,)

In [197]:
rows_idx, cols_idx = rasterio.transform.rowcol(
    transform,
    cloud.X,
    cloud.Y
)

In [198]:
from rasterio.transform import rowcol

rows, cols = rowcol(transform, cloud.X, cloud.Y)

ground_z = dtm[rows, cols]

hag = cloud.Z - ground_z

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [166]:
z_bin = (cloud.Z / 1).astype(int)   # 2-foot bins

In [167]:
x_distance = (x_max - x_min) / width
y_distance = (y_max - y_min) / height

In [168]:
from collections import defaultdict

vertical_bins = defaultdict(set)

for r, c, zb in zip(rows_idx, cols_idx, z_bin):
    vertical_bins[(r, c)].add(zb)

In [169]:
columnarity = np.zeros((height, width), dtype=np.uint16)

for (r, c), zb_set in vertical_bins.items():
    columnarity[r, c] = len(zb_set)


In [170]:
with rasterio.open(
        "columnarity.tif",
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=density.dtype,
        crs=crs,
        transform=transform,
    ) as dst:

        dst.write(columnarity, 1)